In [ ]:
!pip install dagshub
!pip install mlflow 
!pip install dagshub mlflow imbalanced-learn --quiet

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import os
import dagshub
import mlflow
import dagshub.auth

# Set token as environment variable (replace with actual token)
os.environ['DAGSHUB_TOKEN'] = '237c5c1b853b5aee083fa279da4224f289a29cbb'

# Initialize DagsHub
dagshub.auth.add_app_token(token=os.environ['DAGSHUB_TOKEN'])
dagshub.init(repo_owner='slomi23', repo_name='slomi23ML2', mlflow=True)



In [ ]:
import pandas as pd
import numpy as np
train_identity=pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")
train_transaction=pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")

test_identity=pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv")
test_transaction=pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv")

In [ ]:
trainset=pd.merge(train_identity, train_transaction, on="TransactionID", how="left")
trainset.head()

In [ ]:
from sklearn.model_selection import train_test_split

# Assuming your target column is 'isFraud'
X = trainset.drop('isFraud', axis=1)
y = trainset['isFraud']

# Split the data - 80% train, 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X, 
    y, 
    test_size=0.2,  # 20% for validation
    random_state=42,  # For reproducibility
    stratify=y  # Important for classification datasets with imbalanced classes
)

print(f"Training set size: {len(X_train)} samples")
print(f"Validation set size: {len(X_val)} samples")
print(f"Training set fraud percentage: {y_train.mean():.2%}")
print(f"Validation set fraud percentage: {y_val.mean():.2%}")

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
import shap
class TargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, categorical_columns, smoothing=1.0):
        self.categorical_columns = categorical_columns
        self.smoothing = smoothing
        self.target_means = {}
        self.global_mean = None
        
    def fit(self, X, y):
        self.global_mean = y.mean()
        
        for col in self.categorical_columns:
            # Calculate target mean for each category
            cat_counts = X[col].value_counts()
            cat_target_means = y.groupby(X[col]).mean()
            
            # Apply smoothing
            self.target_means[col] = (
                (cat_target_means * cat_counts + self.global_mean * self.smoothing) / 
                (cat_counts + self.smoothing)
            )
        return self
    
    def transform(self, X):
        X_encoded = X.copy()
        
        for col in self.categorical_columns:
            # Map categories to target-encoded values
            X_encoded[col] = X[col].map(self.target_means[col]).fillna(self.global_mean)
        
        return X_encoded

In [ ]:

class SHAPFeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, model, n_features_to_select=10, importance_threshold=0.01):
        self.model = model
        self.n_features_to_select = n_features_to_select
        self.importance_threshold = importance_threshold
        self.selected_features = None
        self.feature_importance = None
        self.selected_feature_indices = None
        self.feature_names = None
    
    def fit(self, X, y):
        # Store feature names if available
        if hasattr(X, 'columns'):
            self.feature_names = X.columns
        else:
            # If X is numpy array, create generic feature names
            self.feature_names = [f'feature_{i}' for i in range(X.shape[1])]
        
        # Train a model to get SHAP values
        self.model.fit(X, y)
        
        # Calculate SHAP values
        explainer = shap.Explainer(self.model, X)
        shap_values = explainer(X)
        
        # Get feature importance from SHAP values
        self.feature_importance = np.abs(shap_values.values).mean(axis=0)
        
        # Select features based on importance threshold
        top_indices = np.where(self.feature_importance > self.importance_threshold)
        self.selected_feature_indices = top_indices[0]
        
        # Store feature names for selected features
        self.selected_features = [self.feature_names[i] for i in self.selected_feature_indices]
        
        return self
    
    def transform(self, X):
        if self.selected_feature_indices is None:
            raise ValueError("Must fit before transform")
        return X[:, self.selected_feature_indices]


In [ ]:
import mlflow
import mlflow.sklearn
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report


# Define categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_cols = [col for col in numerical_cols if col != 'isFraud']
# Remove columns with no values before pipeline
cols_to_remove = ['dist1', 'D11', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11']
X_train = X_train.drop(columns=[col for col in cols_to_remove if col in X_train.columns])
X_val = X_val.drop(columns=[col for col in cols_to_remove if col in X_val.columns])


from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Create the XGBoost pipeline
full_pipeline = Pipeline([
    ('target_encoder', TargetEncoder(categorical_columns=categorical_cols)),
    ('imputer', SimpleImputer(strategy='mean')), 
    ('feature_selector', SHAPFeatureSelector(
        model=LogisticRegression(max_iter=10000, random_state=42), 
        n_features_to_select=100
    )),
    ('scaler', StandardScaler()),
    ('xgboost', XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        eval_metric='logloss',  # For binary classification
        use_label_encoder=False  # To avoid warning
    ))
])
mlflow.set_experiment("XGBoostTargetEncodingShap")
with mlflow.start_run():
    # Log model type and feature selection
    mlflow.log_param("model_type", "xgboost_with_target_encoding_and_shap")
    mlflow.log_param("feature_selection_method", "shap_based")
    mlflow.log_param("categorical_encoding", "target_encoding")
    mlflow.log_param("final_algorithm", "xgboost")
    mlflow.log_param("n_features_selected", 100)
    
    # Log XGBoost specific parameters
    mlflow.log_param("xgboost_n_estimators", 100)
    mlflow.log_param("xgboost_max_depth", 6)
    mlflow.log_param("xgboost_learning_rate", 0.1)
    mlflow.log_param("xgboost_random_state", 42)
    mlflow.log_param("xgboost_eval_metric", "logloss")
    mlflow.log_param("xgboost_use_label_encoder", False)
    
    # Alternative: Log all parameters in a dictionary
    xgb_params = {
        "model_type": "xgboost_with_target_encoding_and_shap",
        "feature_selection_method": "shap_based",
        "categorical_encoding": "target_encoding",
        "final_algorithm": "xgboost",
        "n_features_selected": 100,
        "xgboost_n_estimators": 100,
        "xgboost_max_depth": 6,
        "xgboost_learning_rate": 0.1,
        "xgboost_random_state": 42,
        "xgboost_eval_metric": "logloss",
        "xgboost_use_label_encoder": False
    }
    mlflow.log_params(xgb_params)      
    # Fit the pipeline
    full_pipeline.fit(X_train, y_train)
    y_train_pred = full_pipeline.predict(X_train)
    y_val_pred = full_pipeline.predict(X_val)
    
    y_train_pred_binary = (y_train_pred > 0.5).astype(int)
    y_val_pred_binary = (y_val_pred > 0.5).astype(int)
    
    # Calculate metrics for both sets
    train_accuracy = accuracy_score(y_train, y_train_pred_binary)
    val_accuracy = accuracy_score(y_val, y_val_pred_binary)
    
    # Log all metrics
    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("val_accuracy", val_accuracy)
    mlflow.log_metric("roc_auc", roc_auc)  # Your existing validation ROC AUC
    # Make predictions
    y_pred = full_pipeline.predict(X_val)
    y_pred_binary = (y_pred > 0.5).astype(int)
    
    # Calculate metrics
    accuracy = accuracy_score(y_val, y_pred_binary)
    roc_auc = roc_auc_score(y_val, y_pred)
    
    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("roc_auc", roc_auc)
    
    # Print results
    print("\n=== MODEL RESULTS ===")
    print(f"Training Accuracy: {train_accuracy:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"ROC AUC Score: {roc_auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_val, y_pred_binary))
    
    # Log the pipeline
    mlflow.sklearn.log_model(full_pipeline, "ridge_pipeline_with_shap_and_target_encoding")
    
    # Log feature importance
    feature_selector = full_pipeline.named_steps['feature_selector']
    feature_importance_df = pd.DataFrame({
        'feature': feature_selector.selected_features,
        'importance': feature_selector.feature_importance[np.argsort(feature_selector.feature_importance)[-len(feature_selector.selected_features):]]
    })
    mlflow.log_text(feature_importance_df.to_string(), "feature_importance.txt")
    
    print(f"\nSelected {len(feature_selector.selected_features)} features")
    print("Top 10 most important features:")
    print(feature_importance_df.sort_values('importance', ascending=False).head(10))
    
    # Get run info and display link
    run_info = mlflow.active_run().info
    run_url = f"http://localhost:5000/#/experiments/{run_info.experiment_id}/runs/{run_info.run_id}"
    
    print(f"\nMLflow run completed. Run ID: {run_info.run_id}")
    print(f"View your run here: {run_url}")
    
    # Log the run URL as an artifact for easy access
    mlflow.log_text(run_url, "run_url.txt")
